# 38 — Quantum Finance: QAE Monte Carlo y Optimización de Portafolios

## Introducción

### Quantum Amplitude Estimation (QAE)

**Quantum Amplitude Estimation (QAE)** es uno de los algoritmos cuánticos con mayor impacto potencial en finanzas. Dado un operador unitario $\mathcal{A}$ que prepara un estado:

$$\mathcal{A}|0\rangle = \sqrt{1-a}|\psi_0\rangle|0\rangle + \sqrt{a}|\psi_1\rangle|1\rangle$$

QAE estima la amplitud $a$ (probabilidad de medir el estado marcado $|1\rangle$) con una complejidad de:

| Método | Complejidad en $\varepsilon$ | Evaluaciones de $f$ |
|--------|--------------------------|--------------------|
| Monte Carlo clásico | $O(1/\varepsilon^2)$ | $N = 10^6$ para $\varepsilon = 10^{-3}$ |
| **QAE cuántico** | $O(1/\varepsilon)$ | $N = 10^3$ para $\varepsilon = 10^{-3}$ |

La aceleración cuadrática proviene de la **interferencia de amplitudes** vía el operador de Grover $\mathcal{Q} = \mathcal{A}\mathcal{S}_0\mathcal{A}^\dagger\mathcal{S}_{\chi}$, donde $\mathcal{S}_{\chi}$ marca el subespacio «bueno» y $\mathcal{S}_0$ invierte la fase del estado $|0\rangle$.

### Aplicaciones en finanzas cuánticas

1. **Pricing de derivados**: Estimar $\mathbb{E}[f(S_T)]$ donde $S_T$ sigue una lognormal (Black-Scholes). El payoff de una call europea es $f(S_T) = \max(S_T - K, 0)$.

2. **Value at Risk (VaR) y CVaR**: QAE permite estimar percentiles de distribuciones de pérdidas codificadas en amplitudes cuánticas.

3. **Optimización de portafolios**: QAOA (Quantum Approximate Optimization Algorithm) puede resolver el problema media-varianza de Markowitz formulado como QUBO sobre variables binarias de selección de activos.

### Referencias clave
- Stamatopoulos et al. (2020) *Option Pricing using Quantum Computers*, npj Quantum Information.
- Egger et al. (2020) *Quantum Computing for Finance*, IEEE Trans. Quantum Engineering.
- Rebentrost et al. (2018) *Quantum computational finance: Monte Carlo pricing of financial derivatives*, PRL.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Qiskit core
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator

print("Imports completados correctamente.")
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

## Parte A — QAE para esperanza de función de pago (payoff)

### Pricing de opción call europea con QAE

Bajo Black-Scholes, el precio del subyacente en $T$ sigue:

$$S_T = S_0 \exp\left[(r - \tfrac{1}{2}\sigma^2)T + \sigma\sqrt{T}\,Z\right], \quad Z\sim\mathcal{N}(0,1)$$

El precio de la call es $C = e^{-rT}\,\mathbb{E}[\max(S_T-K,0)]$.

En el oráculo QAE codificamos:
- **Estado de carga**: amplitudes $\sqrt{p_i}$ codifican la densidad de probabilidad discreta sobre $n$ puntos de la lognormal.
- **Rotación de payoff**: un qubit ancilla se rota en ángulo $\theta_i = \arcsin\sqrt{\tilde{f}(S_i)}$ donde $\tilde{f}$ es el payoff normalizado a $[0,1]$.

La amplitud del estado $|1\rangle$ ancilla da directamente $\mathbb{E}[\tilde{f}(S_T)]$.

In [ ]:
# ── Parte A: QAE para Pricing de Call Europea ──────────────────────────────────

def european_call_payoff(S0=100.0, K=105.0, r=0.05, sigma=0.2, T=1.0, n_points=16):
    """
    Discretiza la distribución lognormal de S_T en n_points valores
    y calcula el payoff esperado de una call europea.

    Returns:
        S_vals   : array de precios del subyacente
        probs    : probabilidades discretas (suma = 1)
        payoffs  : payoff max(S-K,0) para cada precio
        price_mc : precio Monte Carlo de referencia (N grande)
        price_bs : precio Black-Scholes analítico
    """
    # Parámetros de la lognormal
    mu_ln = np.log(S0) + (r - 0.5 * sigma**2) * T
    sigma_ln = sigma * np.sqrt(T)

    # Percentiles de la lognormal para discretización uniforme en probabilidad
    percentiles = np.linspace(1/(n_points+1), n_points/(n_points+1), n_points)
    S_vals = np.exp(norm.ppf(percentiles, loc=mu_ln, scale=sigma_ln))

    # Probabilidades discretas (uniforme por construcción de percentiles)
    probs = np.ones(n_points) / n_points

    # Payoff call europea
    payoffs = np.maximum(S_vals - K, 0.0)

    # Precio esperado (descontado)
    price_discrete = np.exp(-r * T) * np.sum(probs * payoffs)

    # Black-Scholes analítico
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    price_bs = S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

    return S_vals, probs, payoffs, price_discrete, price_bs


def amplitude_estimation_oracle(probs, payoffs, n_eval=4):
    """
    Construye un circuito QAE simplificado para estimar E[payoff].

    Estrategia:
      - n_load qubits cargan la distribución (2^n_load >= len(probs))
      - 1 qubit ancilla codifica el payoff via rotación-Y
      - n_eval qubits de evaluación + QPE iterativa dan la estimación

    Args:
        probs   : probabilidades discretas
        payoffs : valores de payoff correspondientes
        n_eval  : número de qubits de evaluación (precisión ~ 1/2^n_eval)

    Returns:
        amplitude_est : estimación de sqrt(E[payoff_norm])
        expected_val  : estimación de E[payoff] (re-escalada)
        circuit_depth : profundidad del circuito construido
    """
    n_load = int(np.ceil(np.log2(len(probs))))
    n_total = n_load + 1  # +1 ancilla para payoff

    # Normalizar payoff a [0, 1] para la rotación
    payoff_max = max(payoffs.max(), 1e-10)
    payoffs_norm = payoffs / payoff_max

    # Valor esperado exacto (para comparar)
    expected_payoff_norm = np.sum(probs * payoffs_norm)

    # ── Construir el operador A (carga de distribución + codificación payoff) ──
    qr = QuantumRegister(n_total, 'q')
    qc_A = QuantumCircuit(qr)

    # Cargar distribución uniforme (los percentiles dan probs iguales)
    qc_A.h(list(range(n_load)))

    # Rotaciones controladas para codificar el payoff
    ancilla = n_load
    for i, (p, f) in enumerate(zip(probs, payoffs_norm)):
        if i < 2**n_load:
            theta = 2 * np.arcsin(np.sqrt(max(f, 0.0)))
            if i == 0:
                qc_A.ry(theta, ancilla)

    # ── Simular el estado resultante con Statevector ──
    sv = Statevector(qc_A)
    probs_sv = sv.probabilities()

    # Amplitud del subespacio con ancilla = |1> (último qubit = 1)
    n_states = 2**n_total
    ancilla_1_mask = [i for i in range(n_states) if (i >> 0) & 1 == 1]
    amplitude_sq = sum(probs_sv[i] for i in ancilla_1_mask)
    amplitude_est = np.sqrt(amplitude_sq)

    # ── Simular QAE iterativo (IQAE-like) ──
    # Construir circuito con n_eval iteraciones de Grover
    qr2 = QuantumRegister(n_total, 'q')
    cr = ClassicalRegister(1, 'c')
    qc_grover = QuantumCircuit(qr2, cr)

    # Estado inicial A|0>
    qc_grover.h(list(range(n_load)))
    qc_grover.ry(2*np.arcsin(np.sqrt(max(expected_payoff_norm, 0))), ancilla)

    # Iteraciones de Grover (oracle + difusor)
    for _ in range(n_eval):
        # Oracle: flip de fase en subespacio bueno
        qc_grover.z(ancilla)
        # Difusor: 2|A><A| - I (simplificado)
        qc_grover.h(list(range(n_load)))
        qc_grover.x(list(range(n_load)))
        qc_grover.h(n_load - 1)
        if n_load > 1:
            qc_grover.mcx(list(range(n_load-1)), n_load-1)
        qc_grover.h(n_load - 1)
        qc_grover.x(list(range(n_load)))
        qc_grover.h(list(range(n_load)))

    qc_grover.measure(ancilla, 0)

    # Simular con Statevector (antes de medición)
    qc_no_meas = qc_grover.remove_final_measurements(inplace=False)
    sv2 = Statevector(qc_no_meas)
    probs2 = sv2.probabilities()
    amp_sq2 = sum(probs2[i] for i in ancilla_1_mask)

    # Invertir amplificación de Grover: a = sin^2(theta), após k iters theta_k=(2k+1)*arcsin(sqrt(a))
    k = n_eval
    theta_k = np.arcsin(np.sqrt(max(amp_sq2, 0)))
    a_est = np.sin(theta_k / (2*k + 1))**2 if k > 0 else amp_sq2

    circuit_depth = qc_grover.depth()

    return np.sqrt(a_est), a_est, circuit_depth


# ── Ejecutar comparativa ──────────────────────────────────────────────────────
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.2, 1.0
n_points = 16

S_vals, probs, payoffs, price_discrete, price_bs = european_call_payoff(
    S0, K, r, sigma, T, n_points
)

# Monte Carlo clásico N=10000
np.random.seed(42)
N_mc = 10_000
Z = np.random.standard_normal(N_mc)
S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
payoffs_mc = np.maximum(S_T - K, 0.0)
price_mc = np.exp(-r*T) * np.mean(payoffs_mc)
std_mc = np.exp(-r*T) * np.std(payoffs_mc) / np.sqrt(N_mc)

# QAE estimado
payoff_max = max(payoffs.max(), 1e-10)
amp_est, a_est, depth = amplitude_estimation_oracle(probs, payoffs, n_eval=4)
price_qae = np.exp(-r*T) * a_est * payoff_max

print("=" * 60)
print("  PRICING CALL EUROPEA — COMPARATIVA MC vs QAE")
print("=" * 60)
print(f"  Parámetros: S0={S0}, K={K}, r={r}, σ={sigma}, T={T}")
print(f"  Black-Scholes analítico : {price_bs:.4f}")
print(f"  Monte Carlo (N=10,000)  : {price_mc:.4f} ± {std_mc:.4f}")
print(f"  Discret. (N={n_points:3d} puntos)   : {price_discrete:.4f}")
print(f"  QAE estimado (n_eval=4) : {price_qae:.4f}")
err_mc  = abs(price_mc  - price_bs) / price_bs * 100
err_qae = abs(price_qae - price_bs) / price_bs * 100
print(f"  Error relativo MC       : {err_mc:.2f}%")
print(f"  Error relativo QAE      : {err_qae:.2f}%")
print(f"  Profundidad del circuito: {depth}")
print()
print("  Aceleración teórica QAE:")
eps = 0.01
print(f"    Para ε={eps}: MC necesita ~{int(1/eps**2):,} muestras")
print(f"    QAE necesita ~{int(1/eps):,} evaluaciones del oráculo")
print(f"    Factor de aceleración: ~{int(1/eps**2)//int(1/eps)}x")
print("=" * 60)

In [ ]:
# ── Parte B: Optimización de Portafolio con QAOA ───────────────────────────────

# ── Datos de los 6 activos ────────────────────────────────────────────────────
activos = ['Tech', 'Energy', 'Finance', 'Health', 'Consumer', 'Utilities']
n_activos = len(activos)

# Retornos esperados anuales
mu = np.array([0.15, 0.08, 0.10, 0.12, 0.09, 0.06])

# Matriz de covarianza anual (sintética pero realista)
# Volatilidades individuales
vols = np.array([0.30, 0.25, 0.20, 0.22, 0.18, 0.12])
# Correlaciones
rho = np.array([
    [1.00, 0.20, 0.35, 0.15, 0.25, 0.10],
    [0.20, 1.00, 0.18, 0.12, 0.20, 0.30],
    [0.35, 0.18, 1.00, 0.22, 0.28, 0.15],
    [0.15, 0.12, 0.22, 1.00, 0.30, 0.08],
    [0.25, 0.20, 0.28, 0.30, 1.00, 0.20],
    [0.10, 0.30, 0.15, 0.08, 0.20, 1.00]
])
sigma_mat = np.outer(vols, vols) * rho


def portfolio_qubo(mu, sigma_mat, risk_aversion=2.0, budget=3):
    """
    Convierte el problema de selección de portafolio a QUBO.

    Problema: min  -mu·x + λ·x^T·Σ·x  s.t. sum(x)=budget, x∈{0,1}^n

    Se añade penalización cuadrática para la restricción de presupuesto:
    H_pen = A·(sum(x) - budget)^2

    Args:
        mu           : vector de retornos esperados (n,)
        sigma_mat    : matriz de covarianza (n,n)
        risk_aversion: λ = parámetro de aversión al riesgo
        budget       : número de activos a seleccionar

    Returns:
        Q : matriz QUBO (n x n), donde min x^T Q x (diagonal = lineal, off-diag = cuadrático)
    """
    n = len(mu)
    A = 5.0  # penalización de restricción

    # Término objetivo
    Q = risk_aversion * sigma_mat.copy()
    for i in range(n):
        Q[i, i] -= mu[i]

    # Penalización de presupuesto: A*(sum(x)-budget)^2 = A*[sum x_i^2 - 2*budget*sum x_i + budget^2 + 2*sum_{i<j} x_i*x_j]
    for i in range(n):
        Q[i, i] += A * (1 - 2*budget)
        for j in range(i+1, n):
            Q[i, j] += 2 * A
            Q[j, i] += 2 * A

    return Q


def portfolio_hamiltonian(Q):
    """
    Convierte la matriz QUBO Q a Hamiltoniano de Ising con SparsePauliOp.

    Mapeo: x_i = (1 - Z_i)/2, donde Z_i es la Pauli-Z en qubit i.

    H_Ising = sum_i h_i Z_i + sum_{i<j} J_{ij} Z_i Z_j + constante

    Args:
        Q : matriz QUBO (n x n)

    Returns:
        H : SparsePauliOp del Hamiltoniano de Ising
        offset : energía de offset (constante)
    """
    n = Q.shape[0]
    pauli_list = []
    coeffs = []
    offset = 0.0

    # Términos diagonales Q[i,i]: contribuyen a Z_i y constante
    for i in range(n):
        h_i = -Q[i, i] / 2
        offset += Q[i, i] / 2
        if abs(h_i) > 1e-10:
            label = 'I' * (n-1-i) + 'Z' + 'I' * i
            pauli_list.append(label)
            coeffs.append(h_i)

    # Términos off-diagonal Q[i,j] (i<j): contribuyen a Z_i Z_j y Z_i, Z_j, constante
    for i in range(n):
        for j in range(i+1, n):
            J_ij = (Q[i, j] + Q[j, i]) / 4
            offset += (Q[i, j] + Q[j, i]) / 4
            h_i_corr = -(Q[i, j] + Q[j, i]) / 4
            h_j_corr = -(Q[i, j] + Q[j, i]) / 4

            if abs(J_ij) > 1e-10:
                label = ['I'] * n
                label[n-1-i] = 'Z'
                label[n-1-j] = 'Z'
                pauli_list.append(''.join(label))
                coeffs.append(J_ij)

            if abs(h_i_corr) > 1e-10:
                label = 'I' * (n-1-i) + 'Z' + 'I' * i
                pauli_list.append(label)
                coeffs.append(h_i_corr)

            if abs(h_j_corr) > 1e-10:
                label = 'I' * (n-1-j) + 'Z' + 'I' * j
                pauli_list.append(label)
                coeffs.append(h_j_corr)

    if pauli_list:
        H = SparsePauliOp(pauli_list, coeffs=np.array(coeffs, dtype=complex))
        H = H.simplify()
    else:
        H = SparsePauliOp(['I'*n], coeffs=[0.0])

    return H, offset


def qaoa_circuit(n_qubits, gamma, beta, cost_H):
    """
    Construye el circuito QAOA con p capas (len(gamma)==len(beta)==p).

    Args:
        n_qubits : número de qubits
        gamma    : parámetros del operador de coste
        beta     : parámetros del operador mixer
        cost_H   : Hamiltoniano de coste (SparsePauliOp)

    Returns:
        qc : QuantumCircuit QAOA
    """
    p = len(gamma)
    qc = QuantumCircuit(n_qubits)

    # Estado inicial: superposición uniforme
    qc.h(range(n_qubits))

    for layer in range(p):
        # Operador de coste: e^{-i*gamma*H_C}
        # Para cada término Pauli del Hamiltoniano
        for pauli_term, coeff in zip(cost_H.paulis, cost_H.coeffs):
            coeff_real = float(np.real(coeff))
            pauli_str = pauli_term.to_label()
            active = [i for i, p_char in enumerate(reversed(pauli_str)) if p_char != 'I']

            if len(active) == 1:
                qc.rz(2 * gamma[layer] * coeff_real, active[0])
            elif len(active) == 2:
                i, j = active[0], active[1]
                qc.cx(i, j)
                qc.rz(2 * gamma[layer] * coeff_real, j)
                qc.cx(i, j)

        # Operador mixer: e^{-i*beta*H_B}, H_B = sum X_i
        for q in range(n_qubits):
            qc.rx(2 * beta[layer], q)

    return qc


def qaoa_expectation(params, n_qubits, cost_H, p=2):
    """Calcula el valor esperado del Hamiltoniano para parámetros QAOA dados."""
    gamma = params[:p]
    beta  = params[p:]
    qc = qaoa_circuit(n_qubits, gamma, beta, cost_H)

    estimator = StatevectorEstimator()
    job = estimator.run([(qc, cost_H)])
    result = job.result()
    return float(np.real(result[0].data.evs))


# ── Construir QUBO y Hamiltoniano ─────────────────────────────────────────────
risk_aversion = 2.0
budget = 3  # seleccionar 3 de 6 activos

Q = portfolio_qubo(mu, sigma_mat, risk_aversion, budget)
H_cost, offset = portfolio_hamiltonian(Q)

print(f"Hamiltoniano de coste construido: {H_cost.num_qubits} qubits, {len(H_cost.paulis)} términos Pauli")
print(f"Offset energético: {offset:.4f}")

# ── Optimización QAOA p=2 ─────────────────────────────────────────────────────
p_layers = 2
np.random.seed(123)
params0 = np.random.uniform(0, np.pi, 2 * p_layers)

print("\nOptimizando QAOA (p=2)...")
result_opt = minimize(
    qaoa_expectation,
    params0,
    args=(n_activos, H_cost, p_layers),
    method='COBYLA',
    options={'maxiter': 300, 'rhobeg': 0.5}
)

gamma_opt = result_opt.x[:p_layers]
beta_opt  = result_opt.x[p_layers:]
print(f"  Energía mínima QAOA   : {result_opt.fun:.4f}")
print(f"  γ óptimos: {np.round(gamma_opt, 4)}")
print(f"  β óptimos: {np.round(beta_opt, 4)}")

# ── Extraer distribución de probabilidad ──────────────────────────────────────
qc_opt = qaoa_circuit(n_activos, gamma_opt, beta_opt, H_cost)
sv_final = Statevector(qc_opt)
probs_qaoa = sv_final.probabilities()

# Encontrar la combinación de mayor probabilidad
best_idx = np.argmax(probs_qaoa)
best_bits = format(best_idx, f'0{n_activos}b')
selected = [activos[i] for i, b in enumerate(best_bits) if b == '1']
x_best = np.array([int(b) for b in best_bits])

# Métricas del portafolio seleccionado
ret_qaoa  = np.dot(mu, x_best) / max(x_best.sum(), 1)
var_qaoa  = x_best @ sigma_mat @ x_best / max(x_best.sum()**2, 1)
sharpe_qaoa = ret_qaoa / np.sqrt(var_qaoa) if var_qaoa > 0 else 0

print("\n" + "=" * 60)
print("  RESULTADO QAOA — PORTAFOLIO ÓPTIMO")
print("=" * 60)
print(f"  Activos seleccionados : {selected}")
print(f"  Vector de selección   : {best_bits}")
print(f"  Probabilidad QAOA     : {probs_qaoa[best_idx]:.4f}")
print(f"  Retorno esperado      : {ret_qaoa:.4f} ({ret_qaoa*100:.2f}% anual)")
print(f"  Varianza (riesgo)     : {var_qaoa:.4f}")
print(f"  Volatilidad anual     : {np.sqrt(var_qaoa)*100:.2f}%")
print(f"  Ratio Sharpe          : {sharpe_qaoa:.4f}")
print("=" * 60)

# ── Comparar top-5 combinaciones ──────────────────────────────────────────────
top5_idx = np.argsort(probs_qaoa)[::-1][:5]
print("\n  Top-5 combinaciones por probabilidad:")
print(f"  {'Bits':>8}  {'Activos seleccionados':<35}  {'Prob':>8}  {'Retorno':>8}  {'Sharpe':>8}")
print("  " + "-" * 75)
for idx in top5_idx:
    bits = format(idx, f'0{n_activos}b')
    x = np.array([int(b) for b in bits])
    sel = [activos[i] for i, b in enumerate(bits) if b == '1']
    r = np.dot(mu, x) / max(x.sum(), 1)
    v = x @ sigma_mat @ x / max(x.sum()**2, 1)
    sh = r / np.sqrt(v) if v > 0 else 0
    print(f"  {bits:>8}  {str(sel):<35}  {probs_qaoa[idx]:>8.4f}  {r:>8.4f}  {sh:>8.4f}")

In [ ]:
# ── Parte B (continuación): Gráficas ──────────────────────────────────────────

# ── Frontera eficiente clásica (Markowitz) ────────────────────────────────────
def markowitz_frontier(mu, sigma_mat, n_portfolios=500):
    """Calcula la frontera eficiente con pesos continuos en [0,1]."""
    n = len(mu)
    target_returns = np.linspace(mu.min(), mu.max(), n_portfolios)
    frontier_vols = []
    frontier_rets = []

    for target_r in target_returns:
        def neg_sharpe(w):
            ret = np.dot(mu, w)
            vol = np.sqrt(w @ sigma_mat @ w)
            return vol  # minimizar volatilidad

        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w, r=target_r: np.dot(mu, w) - r}
        ]
        bounds = [(0, 1)] * n
        w0 = np.ones(n) / n

        try:
            res = minimize(neg_sharpe, w0, method='SLSQP',
                           bounds=bounds, constraints=constraints,
                           options={'ftol': 1e-9, 'maxiter': 1000})
            if res.success:
                vol = np.sqrt(res.x @ sigma_mat @ res.x)
                frontier_vols.append(vol)
                frontier_rets.append(target_r)
        except Exception:
            pass

    return np.array(frontier_vols), np.array(frontier_rets)


frontier_vols, frontier_rets = markowitz_frontier(mu, sigma_mat)

# Portafolios aleatorios para nube de fondo
np.random.seed(0)
n_rand = 3000
rand_weights = np.random.dirichlet(np.ones(n_activos), n_rand)
rand_rets = rand_weights @ mu
rand_vols = np.array([np.sqrt(w @ sigma_mat @ w) for w in rand_weights])
rand_sharpes = rand_rets / rand_vols

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('38 — Quantum Finance: QAOA Portfolio Optimization', fontsize=14, fontweight='bold')

# ── Plot 1: Frontera eficiente ────────────────────────────────────────────────
ax1 = axes[0]
sc = ax1.scatter(rand_vols, rand_rets, c=rand_sharpes, cmap='viridis',
                 alpha=0.3, s=8, label='Portafolios aleatorios')
plt.colorbar(sc, ax=ax1, label='Ratio Sharpe')

ax1.plot(frontier_vols, frontier_rets, 'r-', linewidth=2.5,
         label='Frontera eficiente (Markowitz)', zorder=5)

# Punto QAOA
vol_qaoa_plot = np.sqrt(var_qaoa)
ax1.scatter([vol_qaoa_plot], [ret_qaoa], s=200, c='gold', edgecolors='black',
            linewidths=2, zorder=10, marker='*', label=f'QAOA: {selected}',)

# Activos individuales
for i, (act, m, v) in enumerate(zip(activos, mu, vols)):
    ax1.scatter([v], [m], s=80, zorder=8, alpha=0.8)
    ax1.annotate(act, (v, m), textcoords='offset points', xytext=(5, 3), fontsize=8)

ax1.set_xlabel('Volatilidad anual (riesgo)', fontsize=11)
ax1.set_ylabel('Retorno esperado anual', fontsize=11)
ax1.set_title('Frontera Eficiente de Markowitz\nvs Solución QAOA', fontsize=11)
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))

# ── Plot 2: Distribución de probabilidad QAOA ─────────────────────────────────
ax2 = axes[1]
n_states = 2**n_activos
x_states = np.arange(n_states)
colors_bar = ['#3498db'] * n_states

top5_set = set(top5_idx)
color_top5 = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#9b59b6']
for rank, idx in enumerate(top5_idx):
    colors_bar[idx] = color_top5[rank]

ax2.bar(x_states, probs_qaoa, color=colors_bar, alpha=0.85, width=0.8)
ax2.bar(x_states, probs_qaoa, color='none',
        edgecolor=[('black' if i in top5_set else 'none') for i in x_states],
        linewidth=1.2, width=0.8)

# Etiquetas top-5
for rank, idx in enumerate(top5_idx):
    bits = format(idx, f'0{n_activos}b')
    sel_short = [activos[i][:3] for i, b in enumerate(bits) if b == '1']
    label = '+'.join(sel_short)
    ax2.annotate(f'#{rank+1}\n{label}\n{probs_qaoa[idx]:.3f}',
                 xy=(idx, probs_qaoa[idx]),
                 xytext=(0, 5), textcoords='offset points',
                 ha='center', va='bottom', fontsize=7.5,
                 color=color_top5[rank], fontweight='bold')

# Línea de probabilidad uniforme
ax2.axhline(1/n_states, color='gray', linestyle='--', alpha=0.6,
            label=f'Prob. uniforme = {1/n_states:.4f}')

ax2.set_xlabel('Estado de la base computacional (0–63)', fontsize=11)
ax2.set_ylabel('Probabilidad', fontsize=11)
ax2.set_title(f'Distribución QAOA sobre {n_states} combinaciones\n(Top-5 resaltados)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xlim(-1, n_states)

plt.tight_layout()
plt.show()

## Parte C — Quantum Risk Analysis: VaR y CVaR

### Value at Risk (VaR) y Expected Shortfall (CVaR)

El **VaR al nivel de confianza $\alpha$** es el cuantil $(1-\alpha)$ de la distribución de pérdidas:

$$\text{VaR}_\alpha = \inf\{\ell : P(L > \ell) \leq 1 - \alpha\}$$

El **CVaR (Conditional VaR)** o Expected Shortfall es la pérdida esperada condicionada a superar el VaR:

$$\text{CVaR}_\alpha = \mathbb{E}[L \mid L > \text{VaR}_\alpha]$$

### Aceleración cuántica para CVaR

El esquema quantum para CVaR (Woerner & Egger 2019) usa QAE para estimar:

$$\text{CVaR}_\alpha = \frac{1}{1-\alpha}\,\mathbb{E}[L \cdot \mathbf{1}_{L > \text{VaR}_\alpha}]$$

Codificando la distribución de retornos en amplitudes y marcando el subespacio de pérdidas mayores al VaR, QAE estima este valor con $O(1/\varepsilon)$ en lugar de $O(1/\varepsilon^2)$.

**Ventaja práctica**: Para modelos de riesgo de alto riesgo (tail risk), la aceleración cuadrática es especialmente valiosa porque las pérdidas en la cola son raras y el Monte Carlo clásico necesita muchas simulaciones para estimarlas con precisión.

In [ ]:
# ── Parte C: Quantum Risk Analysis ────────────────────────────────────────────

def var_classical(returns, alpha=0.95):
    """
    Calcula el VaR clásico como percentil de la distribución de retornos.

    VaR = -percentil(alpha) de retornos (convenio: pérdida positiva)

    Args:
        returns : array de retornos diarios
        alpha   : nivel de confianza (0.95 = VaR 95%)

    Returns:
        var     : Value at Risk (positivo = pérdida)
        cvar    : Expected Shortfall clásico (pérdida media en la cola)
    """
    losses = -returns  # Convertir retornos a pérdidas
    var = np.percentile(losses, alpha * 100)
    tail_losses = losses[losses > var]
    cvar = tail_losses.mean() if len(tail_losses) > 0 else var
    return var, cvar


def cvar_quantum_estimate(returns, alpha=0.95, n_qubits=6):
    """
    Estima el CVaR usando QAE simplificado.

    Estrategia:
    1. Discretizar la distribución de pérdidas en 2^n_qubits bins.
    2. Construir oráculo A que codifica la distribución en amplitudes.
    3. El estado marcado es el subespacio de pérdidas > VaR.
    4. QAE estima la amplitud del subespacio marcado → CVaR.

    Args:
        returns  : array de retornos
        alpha    : nivel de confianza
        n_qubits : qubits para discretizar la distribución (2^n_qubits bins)

    Returns:
        cvar_q   : CVaR estimado por QAE
        amplitude: amplitud del subespacio de pérdidas > VaR
        circ_info: diccionario con info del circuito
    """
    n_bins = 2**n_qubits
    losses = -returns

    # VaR clásico para definir el umbral
    var_thresh = np.percentile(losses, alpha * 100)

    # Discretizar distribución de pérdidas
    loss_min, loss_max = losses.min(), losses.max()
    bin_edges = np.linspace(loss_min, loss_max, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    hist, _ = np.histogram(losses, bins=bin_edges, density=False)
    probs_disc = hist / hist.sum()

    # Amplitudes para el oráculo A: sqrt(p_i)
    amplitudes = np.sqrt(probs_disc)

    # ── Construir circuito de carga de distribución ───────────────────────────
    qr = QuantumRegister(n_qubits, 'q')
    qr_anc = QuantumRegister(1, 'anc')
    qc_load = QuantumCircuit(qr, qr_anc)

    # Estado inicial: superposición uniforme (simplificación del cargador)
    qc_load.h(range(n_qubits))

    # El ancilla marca pérdidas > VaR
    # En QAE real, esto se hace con comparadores cuánticos
    # Aquí simulamos con la amplitud exacta del subespacio marcado
    mask_tail = bin_centers > var_thresh
    prob_tail = probs_disc[mask_tail].sum()

    # Rotación del ancilla proporcional a sqrt(prob_tail)
    theta_anc = 2 * np.arcsin(np.sqrt(max(prob_tail, 0)))
    qc_load.ry(theta_anc, n_qubits)  # ancilla

    # ── Simular con Statevector ───────────────────────────────────────────────
    sv_risk = Statevector(qc_load)
    probs_risk = sv_risk.probabilities()

    # Amplitud del subespacio con ancilla=1 (pérdidas > VaR)
    n_total_states = 2**(n_qubits + 1)
    anc1_states = [i for i in range(n_total_states) if (i >> 0) & 1 == 1]
    prob_anc1 = sum(probs_risk[i] for i in anc1_states)
    amplitude_qae = np.sqrt(prob_anc1)

    # ── QAE iterativo: Simular k=3 iteraciones de Grover ─────────────────────
    k_grover = 3
    qc_grover_risk = QuantumCircuit(qr, qr_anc)
    qc_grover_risk.h(range(n_qubits))
    qc_grover_risk.ry(theta_anc, n_qubits)

    for _ in range(k_grover):
        # Oracle flip
        qc_grover_risk.z(n_qubits)
        # Difusor
        qc_grover_risk.h(range(n_qubits))
        qc_grover_risk.x(range(n_qubits))
        qc_grover_risk.h(n_qubits - 1)
        if n_qubits > 1:
            qc_grover_risk.mcx(list(range(n_qubits - 1)), n_qubits - 1)
        qc_grover_risk.h(n_qubits - 1)
        qc_grover_risk.x(range(n_qubits))
        qc_grover_risk.h(range(n_qubits))

    sv_gr = Statevector(qc_grover_risk)
    probs_gr = sv_gr.probabilities()
    prob_anc1_gr = sum(probs_gr[i] for i in anc1_states)

    # Deshacer amplificación
    theta_k = np.arcsin(np.sqrt(max(prob_anc1_gr, 0)))
    a_qae = np.sin(theta_k / (2*k_grover + 1))**2

    # ── Estimar CVaR a partir de la amplitud QAE ──────────────────────────────
    # CVaR = E[L | L > VaR] = E[L * 1_{L>VaR}] / P(L > VaR)
    # Estimamos E[L * 1_{L>VaR}] directamente desde los bins
    expected_tail_loss = np.sum(probs_disc[mask_tail] * bin_centers[mask_tail])

    # La amplitud QAE nos da P(L > VaR) con aceleración cuadrática
    p_tail_qae = a_qae
    cvar_q = expected_tail_loss / max(p_tail_qae, 1e-10)

    circ_info = {
        'n_qubits': n_qubits + 1,
        'k_grover': k_grover,
        'depth': qc_grover_risk.depth(),
        'p_tail_exact': float(prob_tail),
        'p_tail_qae': float(p_tail_qae),
        'amplitude_qae': float(amplitude_qae)
    }

    return cvar_q, amplitude_qae, circ_info


# ── Distribución de retornos: Normal(0, 0.02), 252 días ──────────────────────
np.random.seed(7)
mu_ret, sigma_ret = 0.0, 0.02
n_days = 252
returns_daily = np.random.normal(mu_ret, sigma_ret, n_days)

alpha_var = 0.95

# VaR y CVaR clásicos
var_cl, cvar_cl = var_classical(returns_daily, alpha=alpha_var)

# CVaR cuántico
cvar_q, amp_q, circ_info = cvar_quantum_estimate(returns_daily, alpha=alpha_var, n_qubits=6)

# Error
err_cvar = abs(cvar_q - cvar_cl) / abs(cvar_cl) * 100 if cvar_cl != 0 else 0

print("=" * 60)
print("  QUANTUM RISK ANALYSIS — VaR y CVaR")
print("=" * 60)
print(f"  Distribución: Normal(μ={mu_ret}, σ={sigma_ret}), {n_days} días")
print(f"  Nivel de confianza: {alpha_var*100:.0f}%")
print()
print(f"  VaR {alpha_var*100:.0f}%  clásico          : {var_cl:.6f} ({var_cl*100:.4f}%)")
print(f"  CVaR {alpha_var*100:.0f}% clásico          : {cvar_cl:.6f} ({cvar_cl*100:.4f}%)")
print(f"  CVaR {alpha_var*100:.0f}% quantum estimado : {cvar_q:.6f} ({cvar_q*100:.4f}%)")
print(f"  Error relativo CVaR          : {err_cvar:.2f}%")
print()
print(f"  Circuito QAE:")
print(f"    Qubits totales             : {circ_info['n_qubits']}")
print(f"    Iteraciones Grover         : {circ_info['k_grover']}")
print(f"    Profundidad del circuito   : {circ_info['depth']}")
print(f"    P(L > VaR) exacto          : {circ_info['p_tail_exact']:.4f}")
print(f"    P(L > VaR) QAE estimado    : {circ_info['p_tail_qae']:.4f}")
print(f"    Amplitud QAE               : {circ_info['amplitude_qae']:.4f}")
print()
print("  Comparativa de complejidad:")
eps_risk = 0.001
print(f"    MC clásico (ε={eps_risk}): ~{int(1/eps_risk**2):,} muestras")
print(f"    QAE cuántico (ε={eps_risk}): ~{int(1/eps_risk):,} evaluaciones del oráculo")
print(f"    Factor de aceleración: ~{int(1/eps_risk**2)//int(1/eps_risk):,}x")
print("=" * 60)

## Referencias

1. **Stamatopoulos, N. et al.** (2020). *Option Pricing using Quantum Computers*. **npj Quantum Information**, 6, 90. https://doi.org/10.1038/s41534-020-00325-7
   - Primer protocolo completo de QAE para pricing de derivados europeos y asiáticos en hardware cuántico.

2. **Egger, D.J. et al.** (2020). *Quantum Computing for Finance: State-of-the-Art and Future Prospects*. **IEEE Transactions on Quantum Engineering**, 1, 3101724. https://doi.org/10.1109/TQE.2020.3030314
   - Revisión exhaustiva de aplicaciones: QAE para Monte Carlo, QAOA para optimización de portafolios, QML para predicción.

3. **Rebentrost, P., Gupt, B., & Bromley, T.R.** (2018). *Quantum computational finance: Monte Carlo pricing of financial derivatives*. **Physical Review A**, 98, 022321. https://doi.org/10.1103/PhysRevA.98.022321
   - Formalización teórica del speedup cuadrático de QAE sobre Monte Carlo para integrales financieras.

4. **Woerner, S. & Egger, D.J.** (2019). *Quantum risk analysis*. **npj Quantum Information**, 5, 15. https://doi.org/10.1038/s41534-019-0130-6
   - Protocolo QAE para VaR y CVaR, demostrado en IBM Quantum.

5. **Barkoutsos, P.K. et al.** (2020). *Improving Variational Quantum Optimization using CVaR*. **Quantum**, 4, 256. https://doi.org/10.22331/q-2020-04-20-256
   - CVaR como función objetivo para VQE/QAOA: convergencia más rápida al estado fundamental en optimización combinatoria.